In [1]:
import numpy as np
import torch
from stl import mesh
from matplotlib import pyplot as plt
import os
import torch.nn as nn
from tqdm import tqdm
import json

# Efficient/flash attention не поддерживают backward через backward (нужен для d2v в PINN).
# Math backend поддерживает произвольный порядок градиентов.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

%matplotlib qt

from torch import autograd

def calc_grad(v1, v2, v3, p, x):
    dv1 = autograd.grad(v1.sum(), x, create_graph=True)[0]
    dv2 = autograd.grad(v2.sum(), x, create_graph=True)[0]
    dv3 = autograd.grad(v3.sum(), x, create_graph=True)[0]
    dp = autograd.grad(p.sum(), x, create_graph=True)[0]
    d2v1 = autograd.grad(dv1.sum(), x, create_graph=True)[0]
    d2v2 = autograd.grad(dv2.sum(), x, create_graph=True)[0]
    d2v3 = autograd.grad(dv3.sum(), x, create_graph=True)[0]
    return dv1, dv2, dv3, d2v1, d2v2, d2v3, dp


def calc_res(v1, v2, v3, p, dv1, dv2, dv3, d2v1, d2v2, d2v3, dp):
    mu = 3e-3
    rho = 1050
    res1 = (v1 * dv1[..., 0:1] + v2 * dv1[..., 1:2] + v3 * dv1[..., 2:3]) - mu * (d2v1[..., 0:1] + d2v1[..., 1:2] + d2v1[..., 2:3]) / rho + dp[..., 0:1] / rho
    res2 = (v1 * dv2[..., 0:1] + v2 * dv2[..., 1:2] + v3 * dv2[..., 2:3]) - mu * (d2v2[..., 0:1] + d2v2[..., 1:2] + d2v2[..., 2:3]) / rho + dp[..., 1:2] / rho
    res3 = (v1 * dv3[..., 0:1] + v2 * dv3[..., 1:2] + v3 * dv3[..., 2:3]) - mu * (d2v3[..., 0:1] + d2v3[..., 1:2] + d2v3[..., 2:3]) / rho + dp[..., 2:3] / rho
    res4 = dv1[..., 0:1] + dv2[..., 1:2] + dv3[..., 2:3]
    return [res1, res2, res3, res4]

def mse_zero_loss(f):
    return (f ** 2).mean()

def zero_loss(outputs, n=4):
    loss = 0
    for i in range(n):
        loss += mse_zero_loss(outputs[i])
    loss = loss / n
    return loss

def dist(a, b):
    return ((a - b) ** 2).sum(axis=1) ** 0.5


def lin_seg(x, x_c):
    return dist(x, x_c)


def phi(x, segments, m=3.):
    tmp = 1 / (lin_seg(x, segments[0]) ** m)
    for i in range(1, len(segments)):
        phi_ = lin_seg(x, segments[i])
        tmp = tmp + 1 / (phi_ ** m)
    x = None
    segments = None
    return 1 / (tmp ** (1 / m))

def lin_seg_(x, x_seg):
    # d = ((x_seg[0] - x_seg[1]) ** 2).sum() ** 0.5
    v1 = x_seg[1] - x_seg[0]
    v2 = x_seg[2] - x_seg[0]
    
    normal = np.cross(v1, v2)
    d = np.linalg.norm(normal) / 2
    x_c = x_seg.mean(axis=0)
    f = torch.tensor(np.dot(x - x_seg[0], normal) / d)
    t = (1 / d) * ((d / 2.) ** 2 - dist(x, x_c) ** 2)
    varphi = (t ** 2 + f ** 4) ** 0.5
    tmp = (f ** 2 + (1 / 4.) * (varphi - t) ** 2) ** 0.5
    return tmp

def calc_phi(x, segments):
    phi_seg = phi(x, segments).cpu()
    x = None
    segments = None
    return phi_seg

def get_point_from_segment(points, segment, n, x3=None):
    delta = segment[1] - segment[0]
    for i in range(int(n) + 1):
        if i < int(n) or torch.rand(1) <= (n - int(n)):
            if x3 is not None:
                points.append(torch.cat((segment[0] + delta * torch.rand(1),
                                         x3.reshape(1))))
            else:
                points.append(segment[0] + delta * torch.rand(1))


def sample_boundary_points(segments, m_all, x3=None):
    dist_all = 0
    for i in segments:
        tmp = torch.stack(i, axis=0)
        dist_all += torch.sum(torch.sum((tmp[:, 0] - tmp[:, 1]) ** 2, axis=1) ** 0.5)

    walls_points = []

    for i in range(len(segments)):
        tmp = torch.stack(segments[i], axis=0)
        dist = torch.sum((tmp[:, 0] - tmp[:, 1]) ** 2, axis=1) ** 0.5
        for j in range(len(segments[i])):
            m = dist[j] / (dist_all / m_all)
            get_point_from_segment(walls_points, segments[i][j], m, x3[i].reshape(1) if x3 is not None else x3)

    x = torch.stack(walls_points)
    return x


def is_inside(triangles, X, buffer=False):
    """Copyright 2018 Alexandre Devert

    Permission is hereby granted, free of charge, to any person obtaining a copy of this software and associated documentation files (the "Software"), to deal in the Software without restriction, including without limitation the rights to use, copy, modify, merge, publish, distribute, sublicense, and/or sell copies of the Software, and to permit persons to whom the Software is furnished to do so, subject to the following conditions:

    The above copyright notice and this permission notice shall be included in all copies or substantial portions of the Software."""
    
    # Вычисление определителя 3x3 вдоль оси 1
    def adet(X, Y, Z):
        ret  = (X[:,0] * Y[:,1] * Z[:,2] + Y[:,0] * Z[:,1] * X[:,2] + Z[:,0] * X[:,1] * Y[:,2] - 
                Z[:,0] * Y[:,1] * X[:,2] - Y[:,0] * X[:,1] * Z[:,2] - X[:,0] * Z[:,1] * Y[:,2])
        return ret

    # Инициализация обобщенного порядка точки
    ret = torch.zeros(X.shape[0], dtype=X.dtype).to(X.device)
    
    # Накопление обобщенного порядок точки для каждого треугольника
    for U, V, W in triangles:
        A, B, C = U - X, V - X, W - X
        omega = adet(A, B, C)

        a, b, c = torch.norm(A, dim=1), torch.norm(B, dim=1), torch.norm(C, dim=1)
        k  = a * b * c + c * torch.sum(A * B, dim=1) + a * torch.sum(B * C, dim=1) + b * torch.sum(C * A,dim=1)
        
        ret += torch.arctan2(omega, k)

    return ret >= 2 * np.pi - (buffer if buffer else 0.)


def points_on_triangle(triangle, m):
    p = m % 1
    m = int(np.floor(m)) + (1 if np.random.random() < p else 0)
    x, y = torch.rand(m), torch.rand(m)
    q = abs(x - y)
    s, t, u = q, 0.5 * (x + y - q), 1 - 0.5 * (q + x + y)
    return torch.stack((s * triangle[0] + t * triangle[3] + u * triangle[6],
                        s * triangle[1] + t * triangle[4] + u * triangle[7],
                        s * triangle[2] + t * triangle[5] + u * triangle[8]), 1)


def sample_boundary_points_from_stl(path, centering, max_coord, m_all, return_norm=False):
    mesh_ = mesh.Mesh.from_file(path)

    points = torch.tensor(np.array(mesh_.points))
    # [~np.isclose(mesh_.normals[:, 0], 0., rtol=1e-07, atol=1e-10) & (np.isclose(mesh_.normals[:, 1], 0., rtol=1e-07, atol=1e-10))])

    points[:, :3] -= centering.cpu().numpy()
    points[:, 3:6] -= centering.cpu().numpy()
    points[:, 6:9] -= centering.cpu().numpy()

    points = points / max_coord.cpu().numpy() / 2

    areas = torch.tensor(np.array(mesh_.areas))
    # [~np.isclose(mesh_.normals[:, 0], 0., rtol=1e-07, atol=1e-10) & (np.isclose(mesh_.normals[:, 1], 0., rtol=1e-07, atol=1e-10))])

    areas_all = areas.sum()

    boundary_points = torch.zeros(0, 3)

    for i in range(len(points)):
        m = areas[i] / (areas_all / m_all)

        boundary_points = torch.concatenate((boundary_points,
                                             points_on_triangle(points[i], m)))

    x = boundary_points
    if return_norm:
        norm = torch.tensor(mesh_.normals[0])
        norm = norm / torch.norm(norm)
        return x, norm, areas_all / 1e6
    return x


def load_stl(path, n=64, n_interior=5000000, n_walls=10000, n_inlet=10000, n_outlet=10000, odd=False, length=[1., 1., 1.], device='cpu', use_3d=True, inside_buffer=0.001):
    x_dict = {}
    phi_w_dict = {}
    phi_in_dict = {}
    n_dict = {}
    
    print(f'Mask generation with path: {path}')
    closed_mesh = mesh.Mesh.from_file(path)
    
    centering = torch.zeros(3).to(device)

    closed_points = torch.tensor(np.array(closed_mesh.points)).to(device)

    centering[0] = closed_points[:, ::3].min() + (closed_points[:, ::3].max() - closed_points[:, ::3].min()) / 2
    centering[1] = closed_points[:, 1::3].min() + (closed_points[:, 1::3].max() - closed_points[:, 1::3].min()) / 2
    centering[2] = closed_points[:, 2::3].min() + (closed_points[:, 2::3].max() - closed_points[:, 2::3].min()) / 2

    closed_points[:, :3] -= centering
    closed_points[:, 3:6] -= centering
    closed_points[:, 6:9] -= centering

    max_coord = closed_points.__abs__().max()

    closed_points = closed_points / max_coord / 2

    x1 = torch.linspace(-length[0] / 2, length[0] / 2, n)
    x2 = torch.linspace(-length[1] / 2, length[1] / 2, n) if use_3d else torch.tensor(0.001 * length[1])
    x3 = torch.linspace(-length[2] / 2, length[2] / 2, n)

    x1, x2, x3 = torch.meshgrid(x1, x2, x3, indexing='ij')

    dx = torch.tensor([closed_points[:, ::3].max() - closed_points[:, ::3].min(),
                       closed_points[:, 1::3].max() - closed_points[:, 1::3].min(),
                       closed_points[:, 2::3].max() - closed_points[:, 2::3].min()]).to(device)
    
    x = torch.stack([x1, x2, x3])
    x = x.reshape(3, -1).T.to(device)

    mask = is_inside(zip(closed_points[:, :3], 
                         closed_points[:, 3:6],
                         closed_points[:, 6:9]), x, inside_buffer)
    x_dict['outerior'] = x.cpu()[~mask.cpu()]
    mask = mask.reshape(n, n, n).cpu() if use_3d else mask.reshape(n, n).cpu()
    
    print('done\n\nInterior points generation')
    x = (torch.rand(int(0.2 * n_interior), 3) * dx.cpu() * 1.1 - (dx.cpu() * 1.1 / 2)).to(device)
    mask_ = is_inside(zip(closed_points[:, :3], 
                         closed_points[:, 3:6],
                         closed_points[:, 6:9]), x, inside_buffer)
    
    x = x[mask_]
    x = x.repeat(int(n_interior * 1.3 / len(x)), 1)
    x = x + ((torch.rand(*x.shape).to(device) - 0.5) * dx * 0.05)

    mask_ = is_inside(zip(closed_points[:, :3], 
                         closed_points[:, 3:6],
                         closed_points[:, 6:9]), x, inside_buffer)
    
    x_dict['interior'] = x[mask_].cpu()
    x_dict['interior'] = x_dict['interior'][torch.randperm(len(x_dict['interior']))[:n_interior]]

    closed_mesh = None
    closed_points = closed_points.cpu()
    x1 = None 
    x2 = None
    x3 = None
    dx = None
    x = None
    mask_ = None

    mask = {'num': mask.float(), 'bool': mask}

    print('done\n\nInlet points generation')
    x_dict['inlet'], n_dict['inlet'], s = sample_boundary_points_from_stl(path.replace('.stl', '_1.stl'), centering, max_coord, int(n_inlet * 1.1), return_norm=True)
    print('done\n\nOutlet points generation')
    x_dict['outlet'], n_dict['outlet'], _ = sample_boundary_points_from_stl(path.replace('.stl', '_2.stl'), centering, max_coord, int(n_outlet * 1.1), return_norm=True)
    n_dict['outlet_center'] = x_dict['outlet'].mean(0)
    if odd:
        x_dict['inlet'], x_dict['outlet'] = x_dict['outlet'], x_dict['inlet']
    print('done\n\nWalls points generation')
    x_dict['walls'] = sample_boundary_points_from_stl(path.replace('.stl', '_3.stl'), centering, max_coord, int(n_walls * 1.1))
    print('done\n\n')
    if not use_3d:
        x_dict['interior'] = x_dict['interior'][:, ::2]
        x_dict['walls'] = x_dict['walls'][:, ::2]
        x_dict['inlet'] = x_dict['inlet'][:, ::2]
        x_dict['outlet'] = x_dict['outlet'][:, ::2]
        x_q_fix = x_q_fix[:, ::2]

    x_dict['walls'] = x_dict['walls'][torch.randperm(len(x_dict['walls']))[:n_walls]]
    x_dict['inlet'] = x_dict['inlet'][torch.randperm(len(x_dict['inlet']))[:n_inlet]]
    x_dict['outlet'] = x_dict['outlet'][torch.randperm(len(x_dict['outlet']))[:n_outlet]]

    phi_w_dict['interior'] = calc_phi(x_dict['interior'].to(device), x_dict['walls'].to(device))
    max_phi_w = phi_w_dict['interior'].max()
    phi_w_dict['interior'] = phi_w_dict['interior'] / max_phi_w
    phi_w_dict['outerior'] = - calc_phi(x_dict['outerior'].to(device), x_dict['walls'].to(device)) / max_phi_w
    phi_w_dict['inlet'] = calc_phi(x_dict['inlet'].to(device), x_dict['walls'].to(device)) / max_phi_w
    phi_w_dict['outlet'] = calc_phi(x_dict['outlet'].to(device), x_dict['walls'].to(device)) / max_phi_w
    phi_w_dict['walls'] = torch.zeros(len(x_dict['walls']))

    phi_in_dict['interior'] = calc_phi(x_dict['interior'].to(device), torch.cat((x_dict['walls'], x_dict['inlet'])).to(device))
    max_phi_in = phi_in_dict['interior'].max()
    phi_in_dict['interior'] = phi_in_dict['interior'] / max_phi_in
    phi_in_dict['outerior'] = - calc_phi(x_dict['outerior'].to(device), torch.cat((x_dict['walls'], x_dict['inlet'])).to(device)) / max_phi_in
    phi_in_dict['inlet'] = torch.zeros(len(x_dict['inlet']))
    phi_in_dict['outlet'] = calc_phi(x_dict['outlet'].to(device), torch.cat((x_dict['walls'], x_dict['inlet'])).to(device))  / max_phi_in
    phi_in_dict['walls'] = torch.zeros(len(x_dict['walls']))

    # closed_points_tmp = []

    # for point in closed_points:
    #     if point[2].cpu() < 0.1 and point[2].cpu() > -0.01:
    #         closed_points_tmp.append(torch.stack([point[:3].cpu(), point[3:6].cpu(), point[6:9].cpu()]))
    # # closed_points.shape

    agg_dict = {'mask': mask, 'x_dict': x_dict, 'phi_w_dict': phi_w_dict, 'phi_in_dict': phi_in_dict, 'n_dict': n_dict, 'l': max_coord / 1000, 's': s, 'v_mean': torch.norm(phi_w_dict['inlet'].mean() * n_dict['inlet'])}

    return agg_dict


In [2]:
INTERIOR_SIZE = 500
WALLS_SIZE    = 250
INLET_SIZE    = 100
OUTLET_SIZE   = 100
OUTERIOR_SIZE = 250

Q = 1.5e-6

# Границы срезов, вычисленные из констант датасета
_BND_START = INTERIOR_SIZE                                             # начало boundary (walls)
_BND_END   = INTERIOR_SIZE + WALLS_SIZE + INLET_SIZE + OUTLET_SIZE    # конец non-outerior

B = 2

RESUME_FOR_PINN = True

class Dataset(torch.utils.data.Dataset):
    def __init__(self, path):
        self.data = []
        for dir in os.listdir(path):
            for file in os.listdir(os.path.join(path, dir)):
                if file.count('_') == 1 and file.split('_')[-1] != '-1.stl':
                    self.data.append(load_stl(os.path.join(path, dir, file), device='cuda'))
                    torch.save(self.data[-1]['x_dict']['interior'], os.path.join(path, dir, file).replace('.stl', '.pt'))
                    

    def __getitem__(self, index):
        agg = self.data[index]

        idx_int   = torch.randperm(len(agg['x_dict']['interior']))[:INTERIOR_SIZE]
        idx_w     = torch.randperm(len(agg['x_dict']['walls']))[:WALLS_SIZE]
        idx_in    = torch.randperm(len(agg['x_dict']['inlet']))[:INLET_SIZE]
        idx_out   = torch.randperm(len(agg['x_dict']['outlet']))[:OUTLET_SIZE]
        idx_outer = torch.randperm(len(agg['x_dict']['outerior']))[:OUTERIOR_SIZE]

        x = torch.cat([
            agg['x_dict']['interior'][idx_int],
            agg['x_dict']['walls'][idx_w],
            agg['x_dict']['inlet'][idx_in],
            agg['x_dict']['outlet'][idx_out],
            agg['x_dict']['outerior'][idx_outer],
        ], dim=0)

        # 0=interior, 1=walls, 2=inlet, 3=outlet, 4=outerior
        x_label = torch.cat([
            torch.zeros(INTERIOR_SIZE,             dtype=torch.long),
            torch.ones(WALLS_SIZE,                 dtype=torch.long),
            torch.full((INLET_SIZE,),    2,        dtype=torch.long),
            torch.full((OUTLET_SIZE,),   3,        dtype=torch.long),
            torch.full((OUTERIOR_SIZE,), 4,        dtype=torch.long),
        ])

        phi_w = torch.cat([
            agg['phi_w_dict']['interior'][idx_int],
            agg['phi_w_dict']['walls'][idx_w],
            agg['phi_w_dict']['inlet'][idx_in],
            agg['phi_w_dict']['outlet'][idx_out],
            agg['phi_w_dict']['outerior'][idx_outer],
        ])

        phi_in = torch.cat([
            agg['phi_in_dict']['interior'][idx_int],
            agg['phi_in_dict']['walls'][idx_w],
            agg['phi_in_dict']['inlet'][idx_in],
            agg['phi_in_dict']['outlet'][idx_out],
            agg['phi_in_dict']['outerior'][idx_outer],
        ])

        norm_in    = agg['n_dict']['inlet']
        norm_out   = agg['n_dict']['outlet']
        center_out = agg['n_dict']['outlet_center']

        l = agg['l']
        s = agg['s']
        v_mean = agg['v_mean']

        return x, torch.stack((phi_w, phi_in), 1), torch.cat((norm_out, center_out)), norm_in.repeat(len(x), 1), norm_out.repeat(len(x), 1), center_out.repeat(len(x), 1), l.repeat(len(x), 1), s.repeat(len(x), 1), v_mean.repeat(len(x), 1), x_label

    def __len__(self):
        return len(self.data)

In [3]:
class GAPinn(nn.Module):
    def __init__(self, d_model=512, nhead=8, num_enc_layers=6, num_dec_layers=4,
                 dim_ff=2048, dropout=0.1):
        super().__init__()

        self.projector = nn.Linear(3, d_model)
        self.embedding = nn.Embedding(5, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, norm_first=True, activation='gelu')
        self.transformer_encoder = nn.TransformerEncoder(enc_layer, num_layers=num_enc_layers)

        self.out_head = nn.Linear(d_model, 6)

        dec_layer_phi = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, norm_first=True, activation='gelu')
        self.transformer_decoder_phi = nn.TransformerDecoder(dec_layer_phi, num_layers=num_dec_layers)

        dec_layer_flow = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, norm_first=True)
        self.transformer_decoder_flow = nn.TransformerDecoder(dec_layer_flow, num_layers=num_dec_layers)

        self.phi_head = nn.Linear(d_model, 2)
        self.vel_head = nn.Linear(d_model, 3)
        self.p_head   = nn.Linear(d_model, 1)

    def forward(self, x, norm_in, norm_out, center_out, l, s, v_mean, x_label):
        # x:          (B, N, 3)   N = _BND_END + OUTERIOR_SIZE
        # x_label:    (B, N)  long  {0=interior, 1=walls, 2=inlet, 3=outlet, 4=outerior}
        # norm_in:    (B, 3)
        # norm_out:   (B, 3)
        # center_out: (B, 3)
        B = x.shape[0]

        x_proj = self.projector(x)                 # (B, N, d_model)
        label_emb = self.embedding(x_label)        # (B, N, d_model)
        x_emb  = x_proj + label_emb  # (B, N, d_model)

        # --- Encoder: walls + inlet + outlet  [:, _BND_START:_BND_END] ---
        cls_tokens = self.cls_token.expand(B, -1, -1)                              # (B, 1, d_model)
        enc_in     = torch.cat([cls_tokens, x_emb[:, _BND_START:_BND_END]], dim=1) # (B, N_bnd+1, d_model)
        enc_out    = self.transformer_encoder(enc_in)                               # (B, N_bnd+1, d_model)
        x_cls      = enc_out[:, :1]                                                 # (B, 1, d_model)

        out_pred = self.out_head(x_cls.squeeze(1))                                  # (B, 6)

        # --- Decoder phi: all points ---
        phi_seq  = self.transformer_decoder_phi(x_emb, x_cls)            # (B, N, d_model)
        # phi_seq  = self.transformer_decoder_phi(x_emb + x_cls, enc_out)            # (B, N, d_model)
        phi_pred = self.phi_head(phi_seq)                                            # (B, N, 2)

        # --- Decoder flow: interior + walls + inlet + outlet  [:, :_BND_END] ---
        x_grad = x[:, :_BND_END] * 2 * l[:, :_BND_END]
        x_grad.requires_grad_(True)
        x_proj_grad = self.projector(x_grad / l[:, :_BND_END] / 2)                 # (B, N, d_model)
        x_emb  = x_proj_grad + label_emb[:, :_BND_END]  # (B, N, d_model)

        flow_seq = self.transformer_decoder_flow(
            x_emb[:, :_BND_END], x_cls)                                  # (B, _BND_END, d_model)
            # x_emb[:, :_BND_END] + x_cls, enc_out)                                  # (B, _BND_END, d_model)

        v = (self.vel_head(flow_seq) * phi_pred[:, :_BND_END, 1:2]
                  + phi_pred[:, :_BND_END, 0:1] * norm_in[:, :_BND_END] * ((1 / v_mean[:, :_BND_END])*(Q / s[:, :_BND_END])))      # (B, _BND_END, 3)

        signed_dist = ((x[:, :_BND_END] - center_out[:, :_BND_END])
                       * norm_out[:, :_BND_END]).sum(-1, keepdim=True)
        p = self.p_head(flow_seq) * signed_dist * 10                       # (B, _BND_END, 1)

        return out_pred, phi_pred, v[..., 0:1], v[..., 1:2], v[..., 2:3], p, x_grad

In [4]:
# # Test forward pass (small N for speed)
# N_test = INLET_SIZE + OUTLET_SIZE + WALLS_SIZE + INTERIOR_SIZE + OUTERIOR_SIZE

# model = GAPinn().cuda()
# model.eval()

# x_t         = torch.randn(B, N_test, 3).cuda()
# x_label_t   = torch.randint(0, 4, (B, N_test)).cuda()
# norm_in_t   = torch.randn(B, N_test, 3).cuda();  norm_in_t  /= norm_in_t.norm(dim=-1, keepdim=True)
# norm_out_t  = torch.randn(B, N_test, 3).cuda();  norm_out_t /= norm_out_t.norm(dim=-1, keepdim=True)
# center_out_t = torch.randn(B, N_test, 3).cuda()
# l = torch.randn(B, N_test, 1).cuda()
# s = torch.randn(B, N_test, 1).cuda()
# v_mean = torch.randn(B, N_test, 1).cuda()

# out_pred, phi_pred, p, v1, v2, v3, x_grad = model(x_t, norm_in_t, norm_out_t, center_out_t, l, s, v_mean, x_label_t)

# print('out_pred:', out_pred.shape)   # (B, 6)
# print('phi_pred:', phi_pred.shape)   # (B, N, 2)
# print('p:       ', p.shape)          # (B, N, 1)
# print('v1:       ', v1.shape)          # (B, N, 1)
# print('v2:       ', v2.shape)          # (B, N, 1)
# print('v3:       ', v3.shape)          # (B, N, 1)

# dv1, dv2, dv3, d2v1, d2v2, d2v3, dp = calc_grad(v1, v2, v3, p, x_grad)

# res = calc_res(v1, v2, v3, p, dv1, dv2, dv3, d2v1, d2v2, d2v3, dp)

In [5]:
# import matplotlib.pyplot as plt
# import numpy as np

# # Fixing random state for reproducibility
# np.random.seed(196808001)


# def randrange(n, vmin, vmax):
#     """
#     Helper function to make an array of random numbers having shape (n, )
#     with each number distributed Uniform(vmin, vmax).
#     """
#     return (vmax - vmin)*np.random.rand(n) + vmin

# fig = plt.figure()
# ax = fig.add_subplot(projection='3d')

# # ax.scatter(*x_outlet.T, s=0.9, c=phi_seg['all'][list(phi_seg['all'].keys())[0]].cpu())
# ax.scatter(*x_dict['interior'].T, s=0.9, c=phi_in_dict['interior'].cpu())
# # ax.scatter(*x_interior[(x_interior[:, 2] > -0.001) & (x_interior[:, 2] < 0.001)].T, s=0.9, c=phi_seg['all'][list(phi_seg['all'].keys())[0]])

# ax.set_xlabel('X Label')
# ax.set_ylabel('Y Label')
# ax.set_zlabel('Z Label')
# # plt.colorbar()
# plt.show()

In [6]:
# import matplotlib.pyplot as plt
# import numpy as np

# # Fixing random state for reproducibility
# np.random.seed(196808001)


# def randrange(n, vmin, vmax):
#     """
#     Helper function to make an array of random numbers having shape (n, )
#     with each number distributed Uniform(vmin, vmax).
#     """
#     return (vmax - vmin)*np.random.rand(n) + vmin

# fig = plt.figure()
# ax = fig.add_subplot(projection='3d')

# ax.scatter(*x_outlet[:, :2].T, phi_seg['all'][list(phi_seg['all'].keys())[0]].cpu(), s=0.5, c=phi_seg['all'][list(phi_seg['all'].keys())[0]].cpu())
# # ax.scatter(*x_interior[(x_interior[:, 2] > -0.001) & (x_interior[:, 2] < 0.001)][:, :2].T, phi_seg['all'][list(phi_seg['all'].keys())[0]], s=0.5, c=phi_seg['all'][list(phi_seg['all'].keys())[0]])

# ax.set_xlabel('X Label')
# ax.set_ylabel('Y Label')
# ax.set_zlabel('Z Label')
# # plt.colorbar()
# plt.show()

In [98]:
ga_pinn = GAPinn().cuda()

if RESUME_FOR_PINN:
    # ga_pinn.load_state_dict(torch.load('mlp_pinn.pth'))
    ga_pinn.load_state_dict(torch.load('mlp_pinn_resume.pth'))

# dataset = Dataset('G:/SimVascDataset new/')

loader = torch.utils.data.DataLoader(dataset, batch_size=B, shuffle=True)

if RESUME_FOR_PINN:
    optimizer = torch.optim.Adam(ga_pinn.transformer_decoder_flow.parameters(), lr=1e-4)
else:
    optimizer = torch.optim.Adam(ga_pinn.parameters(), lr=1e-4)

    # optimizer.load_state_dict(torch.load('optimizer_resume.pth'))

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 50, 0.97,
                                                        last_epoch=- 1)
loss_fcn = torch.nn.MSELoss()

if RESUME_FOR_PINN:
    with open('history_resume.json', 'r') as fp:
        history = json.load(fp)
else:
    history = {'res_1': [], 'res_2': [], 'res_3': [], 'res_4': [], 'mse_out': [], 'mse_phi': []}

for i in tqdm(range(10000)):
    ga_pinn.train()
    for x, phi, out, norm_in, norm_out, center_out, l, s, v_mean, x_label in tqdm(loader):
        optimizer.zero_grad()

        x, phi, out, norm_in, norm_out, center_out, l, s, v_mean, x_label = \
        x.to('cuda'), phi.to('cuda'), out.to('cuda'), norm_in.to('cuda'), norm_out.to('cuda'), center_out.to('cuda'), l.to('cuda'), s.to('cuda'), v_mean.to('cuda'), x_label.to('cuda')

        out_pred, phi_pred, v1, v2, v3, p, x_grad = ga_pinn(x, norm_in, norm_out, center_out, l, s, v_mean, x_label)

        

        if RESUME_FOR_PINN or (i > 1000):
            if (not RESUME_FOR_PINN) and (i == 1001):
                optimizer = torch.optim.Adam(ga_pinn.transformer_decoder_flow.parameters(), lr=5e-4)
                lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 10, 0.97,
                                                                        last_epoch=- 1)

            dv1, dv2, dv3, d2v1, d2v2, d2v3, dp = calc_grad(v1, v2, v3, p, x_grad)

            res = calc_res(v1, v2, v3, p, dv1, dv2, dv3, d2v1, d2v2, d2v3, dp)

            loss_res = zero_loss(res)
            
            loss = loss_res
        else:
            loss_out = loss_fcn(out_pred, out)

            loss_phi = loss_fcn(phi_pred, phi)

            loss = loss_out + loss_phi

        loss.backward()

        optimizer.step()

        if RESUME_FOR_PINN or i > 1000:
            history['res_1'].append(mse_zero_loss(res[0].detach().cpu()).item())
            history['res_2'].append(mse_zero_loss(res[1].detach().cpu()).item())
            history['res_3'].append(mse_zero_loss(res[2].detach().cpu()).item())
            history['res_4'].append(mse_zero_loss(res[3].detach().cpu()).item())
        else:
            history['mse_out'].append(loss_out.detach().cpu().item())
            history['mse_phi'].append(loss_phi.detach().cpu().item())
    
    if RESUME_FOR_PINN or (i > 1000):
        torch.save(ga_pinn.state_dict(), f'mlp_pinn_resume.pth')
        torch.save(optimizer.state_dict(), f'optimizer_resume.pth')
        with open('history_resume.json', 'w') as fp:
            json.dump(history, fp)
    else:
        torch.save(ga_pinn.state_dict(), f'mlp_pinn.pth')
        torch.save(optimizer.state_dict(), f'optimizer.pth')

        with open('history.json', 'w') as fp:
            json.dump(history, fp)

    lr_scheduler.step()

C:\Users\chest\AppData\Local\Temp\ipykernel_33252\1661249308.py:13: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(enc_layer, num_layers=num_enc_layers)
 20%|██        | 2020/10000 [8:27:07<33:23:23, 15.06s/it]


KeyboardInterrupt: 

In [75]:
ga_pinn = GAPinn().cuda()

ga_pinn.load_state_dict(torch.load('Новая папка/mlp_pinn_resume.pth'))

C:\Users\chest\AppData\Local\Temp\ipykernel_33252\1661249308.py:13: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(enc_layer, num_layers=num_enc_layers)


<All keys matched successfully>

In [100]:
# with open('history_resume.json', 'r') as fp:
#     history = json.load(fp)

with torch.no_grad():
    ga_pinn.eval()
    for x, phi, out, norm_in, norm_out, center_out, l, s, v_mean, x_label in tqdm(loader):
        optimizer.zero_grad()

        x, phi, out, norm_in, norm_out, center_out, l, s, v_mean, x_label = \
        x.to('cuda'), phi.to('cuda'), out.to('cuda'), norm_in.to('cuda'), norm_out.to('cuda'), center_out.to('cuda'), l.to('cuda'), s.to('cuda'), v_mean.to('cuda'), x_label.to('cuda')

        out_pred, phi_pred, v1, v2, v3, p, x_grad = ga_pinn(x, norm_in, norm_out, center_out, l, s, v_mean, x_label)
        break

  0%|          | 0/9 [00:00<?, ?it/s]


In [99]:
plt.plot(history['res_1'])
plt.yscale('log')
plt.show()

# plt.plot(history['res_2'])
# plt.yscale('log')
# plt.show()

# plt.plot(history['res_3'])
# plt.yscale('log')
# plt.show()

plt.plot(history['res_4'])
plt.yscale('log')
plt.show()

# # plt.plot(history['mse_out'])
# # plt.yscale('log')
# # plt.show()

# plt.plot(history['mse_phi'])
# plt.yscale('log')
# plt.show()

In [12]:
len(dataset)

18

In [89]:
idx = 1

In [14]:
signed_dist = ((x[:, :_BND_END] - center_out[:, :_BND_END])
                       * norm_out[:, :_BND_END]).sum(-1, keepdim=True)

In [15]:
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    initial_lr: 0.0001
    lr: 2.2612410099957642e-07
    maximize: False
    weight_decay: 0
)

In [101]:
import matplotlib.pyplot as plt
import numpy as np

# Fixing random state for reproducibility
np.random.seed(196808001)


def randrange(n, vmin, vmax):
    """
    Helper function to make an array of random numbers having shape (n, )
    with each number distributed Uniform(vmin, vmax).
    """
    return (vmax - vmin)*np.random.rand(n) + vmin

fig = plt.figure()
ax = fig.add_subplot(projection='3d')

ax.scatter(*x[idx, :-250].cpu().T, s=20, c=res[3][idx, :].detach().cpu().__abs__())

ax.set_xlabel('X Label')
ax.set_ylabel('Y Label')
ax.set_zlabel('Z Label')
# plt.colorbar()
plt.show()

In [102]:
res[3][idx, :].detach().cpu().__abs__().max()

tensor(6.0817)

In [103]:
import matplotlib.pyplot as plt
import numpy as np

# Fixing random state for reproducibility
np.random.seed(196808001)


def randrange(n, vmin, vmax):
    """
    Helper function to make an array of random numbers having shape (n, )
    with each number distributed Uniform(vmin, vmax).
    """
    return (vmax - vmin)*np.random.rand(n) + vmin

fig = plt.figure()
ax = fig.add_subplot(projection='3d')


ax.scatter(*x[idx, :-250].cpu().T, s=20, c=((v1[idx, :, 0].detach().cpu() ** 2 + v2[idx, :, 0].detach().cpu() ** 2 + v3[idx, :, 0].detach().cpu() ** 2) ** 0.5).detach().cpu())

ax.set_xlabel('X Label')
ax.set_ylabel('Y Label')
ax.set_zlabel('Z Label')
# plt.colorbar()
plt.show()

In [105]:
import matplotlib.pyplot as plt
import numpy as np

np.random.seed(196808001)

fig = plt.figure()
ax = fig.add_subplot(projection='3d')

pos = x[idx, :-250].cpu()
u = v1[idx, :, 0].detach().cpu()
v = v2[idx, :, 0].detach().cpu()
w = v3[idx, :, 0].detach().cpu()

speed = (u ** 2 + v ** 2 + w ** 2) ** 0.5

# ax.scatter(*pos.T, s=20, c=speed)

# Subsample to avoid overcrowding
step = 1
pos_q = pos[::step]
u_q, v_q, w_q = u[::step], v[::step], w[::step]

scale = float(pos.abs().max()) * 0.05
ax.quiver(
    pos_q[:, 0], pos_q[:, 1], pos_q[:, 2],
    u_q, v_q, w_q,
    length=scale,
    color='r', alpha=0.8, linewidth=0.7, arrow_length_ratio=0.7,
)

ax.set_xlabel('X Label')
ax.set_ylabel('Y Label')
ax.set_zlabel('Z Label')
plt.show()

In [50]:
import matplotlib.pyplot as plt
import numpy as np

# Fixing random state for reproducibility
np.random.seed(196808001)


def randrange(n, vmin, vmax):
    """
    Helper function to make an array of random numbers having shape (n, )
    with each number distributed Uniform(vmin, vmax).
    """
    return (vmax - vmin)*np.random.rand(n) + vmin

fig = plt.figure()
ax = fig.add_subplot(projection='3d')

ax.scatter(*x[idx, :-250].cpu().T, s=5, c=v2[idx, :, 0].detach().cpu())

ax.set_xlabel('X Label')
ax.set_ylabel('Y Label')
ax.set_zlabel('Z Label')
# plt.colorbar()
plt.show()

In [21]:
p.max()

tensor(25.1141, device='cuda:0')

In [37]:
plt.plot(phi[idx, :, 0].cpu())
plt.plot(phi_pred[idx, :, 0].detach().cpu())
plt.plot(x_label[idx, :].detach().cpu())

TypeError: Axes3D.plot() missing 1 required positional argument: 'ys'

In [ ]:
plt.plot(phi[idx, :, 1].cpu())
plt.plot(phi_pred[idx, :, 1].detach().cpu())
plt.plot(x_label[idx, :].detach().cpu())

In [ ]:
plt.plot(v1[idx, :, 0].detach().cpu())
plt.plot(x_label[idx, :].detach().cpu())

In [39]:
plt.plot(phi[idx, :, 0].cpu())
plt.plot(((v1[idx, :, 0].detach().cpu() ** 2 + v2[idx, :, 0].detach().cpu() ** 2 + v3[idx, :, 0].detach().cpu() ** 2) ** 0.5) / ((1 / v_mean[idx, :_BND_END, 0].detach().cpu())*(Q / s[idx, :_BND_END, 0].detach().cpu())))
plt.plot(x_label[idx, :].detach().cpu())

In [36]:
plt.plot(phi[idx, :, 0].cpu())
a = ((1 / v_mean[idx, :_BND_END, 0].detach().cpu())*(Q / s[idx, :_BND_END, 0].detach().cpu()))
plt.plot(((v1[idx, :, 0].detach().cpu() / a / 3 / norm_in[idx, :-250, 0].cpu() + v2[idx, :, 0].detach().cpu() / a / 3 / norm_in[idx, :-250, 1].cpu() + v3[idx, :, 0].detach().cpu() / a / 3 / norm_in[idx, :-250, 2].cpu())))
plt.plot(x_label[idx, :].detach().cpu())

TypeError: Axes3D.plot() missing 1 required positional argument: 'ys'

In [ ]:
plt.plot(phi[idx, :, 0].cpu())
plt.plot(p[idx, :, 0].detach().cpu() / 2500)
plt.plot(x_label[idx, :].detach().cpu())

In [ ]:
import torch
import os
import torch.nn as nn
from tqdm import tqdm
import json
from modules import *
from clearml import Task, Dataset


# Efficient/flash attention не поддерживают backward через backward (нужен для d2v в PINN).
# Math backend поддерживает произвольный порядок градиентов.
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(False)
torch.backends.cuda.enable_math_sdp(True)

LOCAL = True

if not LOCAL:
    task = Task.init()

    dataset = Dataset.get(dataset_name='SimVascDataset', dataset_project='kornaeva-rnf/GA_PINN_3D')
    DATASET_PATH = dataset.get_local_copy()

    dataset = Dataset.get(dataset_name='trained_models', dataset_project='kornaeva-rnf/GA_PINN_3D')
    MODELS_PATH = dataset.get_local_copy()

    del dataset

else:
    MODELS_PATH = 'trained_models'
    DATASET_PATH = 'SimVascDataset'

INTERIOR_SIZE = 500
WALLS_SIZE    = 250
INLET_SIZE    = 100
OUTLET_SIZE   = 100
OUTERIOR_SIZE = 250

Q = 1.5e-6

# Границы срезов, вычисленные из констант датасета
_BND_START = INTERIOR_SIZE                                             # начало boundary (walls)
_BND_END   = INTERIOR_SIZE + WALLS_SIZE + INLET_SIZE + OUTLET_SIZE    # конец non-outerior

B = 4

TRAIN_PINN = True
RESUME_PINN = False
GEN_INTERIOR_POINTS = True

class Dataset(torch.utils.data.Dataset):
    def __init__(self, path):
        self.data = []
        for dir in os.listdir(path):
            for file in os.listdir(os.path.join(path, dir)):
                if (file.count('_') == 1) and (file.split('_')[-1] != '-1.stl') and ('.stl' in file):
                    if file.replace('.stl', '.pt') not in os.listdir(os.path.join(path, dir)):
                        self.data.append(load_stl(os.path.join(path, dir, file), device='cuda', gen_int_p=GEN_INTERIOR_POINTS))

    def __getitem__(self, index):
        agg = self.data[index]

        idx_int   = torch.randperm(len(agg['x_dict']['interior']))[:INTERIOR_SIZE]
        idx_w     = torch.randperm(len(agg['x_dict']['walls']))[:WALLS_SIZE]
        idx_in    = torch.randperm(len(agg['x_dict']['inlet']))[:INLET_SIZE]
        idx_out   = torch.randperm(len(agg['x_dict']['outlet']))[:OUTLET_SIZE]
        idx_outer = torch.randperm(len(agg['x_dict']['outerior']))[:OUTERIOR_SIZE]

        x = torch.cat([
            agg['x_dict']['interior'][idx_int],
            agg['x_dict']['walls'][idx_w],
            agg['x_dict']['inlet'][idx_in],
            agg['x_dict']['outlet'][idx_out],
            agg['x_dict']['outerior'][idx_outer],
        ], dim=0)

        # 0=interior, 1=walls, 2=inlet, 3=outlet, 4=outerior
        x_label = torch.cat([
            torch.zeros(INTERIOR_SIZE,             dtype=torch.long),
            torch.ones(WALLS_SIZE,                 dtype=torch.long),
            torch.full((INLET_SIZE,),    2,        dtype=torch.long),
            torch.full((OUTLET_SIZE,),   3,        dtype=torch.long),
            torch.full((OUTERIOR_SIZE,), 4,        dtype=torch.long),
        ])

        phi_w = torch.cat([
            agg['phi_w_dict']['interior'][idx_int],
            agg['phi_w_dict']['walls'][idx_w],
            agg['phi_w_dict']['inlet'][idx_in],
            agg['phi_w_dict']['outlet'][idx_out],
            agg['phi_w_dict']['outerior'][idx_outer],
        ])

        phi_in = torch.cat([
            agg['phi_in_dict']['interior'][idx_int],
            agg['phi_in_dict']['walls'][idx_w],
            agg['phi_in_dict']['inlet'][idx_in],
            agg['phi_in_dict']['outlet'][idx_out],
            agg['phi_in_dict']['outerior'][idx_outer],
        ])

        norm_in    = agg['n_dict']['inlet']
        norm_out   = agg['n_dict']['outlet']
        center_out = agg['n_dict']['outlet_center']

        l = agg['l']
        s = agg['s']
        v_mean = agg['v_mean']

        return x, torch.stack((phi_w, phi_in), 1), torch.cat((norm_out, center_out)), norm_in.repeat(len(x), 1), norm_out.repeat(len(x), 1), center_out.repeat(len(x), 1), l.repeat(len(x), 1), s.repeat(len(x), 1), v_mean.repeat(len(x), 1), x_label

    def __len__(self):
        return len(self.data)

In [ ]:
dataset = Dataset(DATASET_PATH)